PROYECTO DESAMBIGUACIÓN DEL SENTIDO DE LAS PALABRAS USANDO NLTK Y APRENDIZAJE AUTOMÁTICO SUPERVISADO

DESARROLLO DEL PROYECTO

Parte 1: Análisis del Corpus Senseval 2

En esta sección se realizó un análisis exploratorio del corpus etiquetado Senseval 2, con el objetivo de comprender la estructura de los datos, la distribución de sentidos para las palabras ambiguas “hard” y “serve”, sus formas gramaticales y la calidad del formato de las instancias.

Resultados obtenidos:

Sentidos y frecuencias:
Hard: HARD1 (3455), HARD2 (502), HARD3 (376)
Serve: SERVE10 (1814), SERVE12 (1272), SERVE2 (853), SERVE6 (439)

Formas gramaticales encontradas:
Hard: aparece como hard, harder, hardest, entre otras variaciones.
Serve: aparece como serve, served, serves, además de algunas formas relacionadas como preserve, observe, reserve.

Instancias mal formateadas:
Hard: 19 instancias incorrectas.
Serve: 76 instancias incorrectas.

En todos los casos, el problema se debe a elementos no estructurados como el token FRASL, que rompe el formato de contexto esperado.

In [12]:
import nltk
nltk.download('senseval')
from nltk.corpus import senseval

# Mostrar las palabras ambiguas disponibles en el corpus
print("Palabras disponibles en el corpus Senseval 2:")
print(senseval.fileids())


[nltk_data] Downloading package senseval to
[nltk_data]     C:\Users\gran_\AppData\Roaming\nltk_data...
[nltk_data]   Package senseval is already up-to-date!
Palabras disponibles en el corpus Senseval 2:
['hard.pos', 'interest.pos', 'line.pos', 'serve.pos']


Parte 2: Contar los sentidos disponibles y la cantidad de instancias por cada sentido

Este código presentado utiliza Python y la biblioteca NLTK (Natural Language Toolkit), junto con el módulo Counter de la biblioteca estándar collections, para analizar el corpus Senseval 2. Su propósito es cargar las instancias de las palabras ambiguas "hard" y "serve" desde los archivos hard.pos y serve.pos, contar la frecuencia de cada sentido asociado a estas palabras, y mostrar los resultados. Este paso es fundamental en el laboratorio para entender la distribución de los sentidos en el corpus, lo cual es clave para tareas posteriores como el entrenamiento de clasificadores Naive Bayes.

In [13]:
from collections import Counter

# Obtener las instancias de las palabras
hard_instances = senseval.instances('hard.pos')
serve_instances = senseval.instances('serve.pos')

# Contar los sentidos para cada palabra
hard_senses = Counter()
for inst in hard_instances:
    hard_senses.update(inst.senses)

serve_senses = Counter()
for inst in serve_instances:
    serve_senses.update(inst.senses)

print("Sentidos para la palabra 'hard':")
for sense, count in hard_senses.items():
    print(f"{sense}: {count} instancias")

print("\nSentidos para la palabra 'serve':")
for sense, count in serve_senses.items():
    print(f"{sense}: {count} instancias")


Sentidos para la palabra 'hard':
HARD1: 3455 instancias
HARD2: 502 instancias
HARD3: 376 instancias

Sentidos para la palabra 'serve':
SERVE10: 1814 instancias
SERVE12: 1272 instancias
SERVE2: 853 instancias
SERVE6: 439 instancias


Parte 3: Identificación de formas gramaticales

Este código define una función que analiza las formas gramaticales de "hard" y "serve" en el corpus Senseval 2, contando las frecuencias de cada combinación de forma léxica y etiqueta POS. La salida revela:

•	"hard": Principalmente adjetivo (JJ, JJR, JJS), con errores de etiquetado (ej. "harder" como JJ) y formas raras (ej. "hardaway").
•	"serve": Predominantemente verbal (VB, VBD, VBZ, VBN, VBP), pero con formas irrelevantes (ej. "preserve", "observe") y errores como "32serves".

Este análisis es clave para el laboratorio, ya que identifica la diversidad gramatical, detecta inconsistencias en el corpus, y proporciona información para enriquecer las características de los clasificadores Naive Bayes. Las limitaciones observadas (etiquetado incorrecto, formas irrelevantes) destacan la necesidad de preprocesamiento adicional.

In [14]:
from collections import Counter

def formas_gramaticales(instances, palabra_base):
    formas = Counter()
    for inst in instances:
        context = list(inst.context)  # aseguramos que es una lista de tuplas
        for elemento in context:
            if isinstance(elemento, tuple) and len(elemento) == 2:
                word, tag = elemento
                if palabra_base in word.lower():
                    formas[(word.lower(), tag)] += 1
    return formas

print("Formas gramaticales de 'hard':")
formas_hard = formas_gramaticales(hard_instances, 'hard')
for forma, count in formas_hard.items():
    print(f"{forma}: {count} veces")

print("\nFormas gramaticales de 'serve':")
formas_serve = formas_gramaticales(serve_instances, 'serve')
for forma, count in formas_serve.items():
    print(f"{forma}: {count} veces")


Formas gramaticales de 'hard':
('hard', 'JJ'): 3706 veces
('harder', 'JJ'): 501 veces
('hardest', 'JJ'): 140 veces
('die-hard', 'JJ'): 1 veces
('harder', 'NNP'): 5 veces
('hardaway', 'NNP'): 1 veces
('harder', 'JJR'): 1 veces

Formas gramaticales de 'serve':
('serve', 'VB'): 1983 veces
('preserve', 'VB'): 2 veces
('served', 'VBD'): 1426 veces
('serves', 'VBZ'): 706 veces
('served', 'VBN'): 68 veces
('serve', 'VBP'): 14 veces
('reserved', 'VBN'): 12 veces
('reserve', 'NNP'): 8 veces
('server', 'NN'): 2 veces
('reserve', 'NN'): 7 veces
('observed', 'VBN'): 2 veces
('deserved', 'VBD'): 2 veces
('32serves', 'NNS'): 1 veces
('observe', 'VB'): 3 veces
('reserves', 'NNS'): 4 veces
('deserve', 'VBP'): 3 veces
('observer', 'NN'): 2 veces
('observers', 'NNS'): 2 veces
('observed', 'VBD'): 2 veces


Parte 4: Validación del formato de las instancias

El código define una función que verifica el formato de las instancias de "hard" y "serve" en el corpus Senseval 2, detectando errores en los campos esenciales o en el formato del contexto. La salida muestra:

•	"hard": 19 instancias mal formateadas (0.44% del total), todas con "FRASL" en el contexto.
•	"serve": 76 instancias mal formateadas (1.74% del total), también con "FRASL".

Este análisis es crucial para la limpieza de datos en el laboratorio, confirmando la presencia de inconsistencias en el corpus (como se mencionó previamente) y destacando la necesidad de preprocesamiento adicional para garantizar la calidad de los datos antes de entrenar los clasificadores Naive Bayes.

In [15]:
def verificar_formato(instances, palabra):
    errores = []
    for i, inst in enumerate(instances):
        if not hasattr(inst, 'word') or not hasattr(inst, 'context') or not hasattr(inst, 'senses') or not hasattr(inst, 'position'):
            errores.append((i, "Faltan campos"))
            continue
        for elemento in inst.context:
            if not (isinstance(elemento, tuple) and len(elemento) == 2):
                errores.append((i, f"Contexto mal formateado: {elemento}"))
                break
    print(f"Instancias mal formateadas para '{palabra}': {len(errores)} encontradas.")
    for i, error in errores[:5]:  # muestra solo los primeros 5
        print(f"Instancia {i}: {error}")
    return errores

errores_hard = verificar_formato(hard_instances, 'hard')
errores_serve = verificar_formato(serve_instances, 'serve')


Instancias mal formateadas para 'hard': 19 encontradas.
Instancia 21: Contexto mal formateado: FRASL
Instancia 110: Contexto mal formateado: FRASL
Instancia 196: Contexto mal formateado: FRASL
Instancia 892: Contexto mal formateado: FRASL
Instancia 999: Contexto mal formateado: FRASL
Instancias mal formateadas para 'serve': 76 encontradas.
Instancia 18: Contexto mal formateado: FRASL
Instancia 24: Contexto mal formateado: FRASL
Instancia 133: Contexto mal formateado: FRASL
Instancia 150: Contexto mal formateado: FRASL
Instancia 151: Contexto mal formateado: FRASL


Parte 5: Extracción de características basada en palabras vecinas (Bag of Words) - Filtrado de palabras (limpieza y stopwords)

El código genera un vocabulario de las 250 palabras más frecuentes en los contextos de "hard" en el corpus Senseval 2, filtrando stopwords, signos de puntuación, formas de "hard", y otros términos de ruido. La salida (['may', 'lose', 'popular', 'support', 'someone', 'kill', 'defeat', 'clever', 'white', 'house']) muestra las primeras 10 palabras, que son razonablemente relevantes, pero incluyen algunos términos generales. Este vocabulario es clave para el modelo de bolsa de palabras en el laboratorio.

In [16]:
from nltk.corpus import stopwords
import string
from nltk import FreqDist
import nltk
nltk.download('stopwords')

# Stopwords y símbolos a eliminar
stop_words = set(stopwords.words('english')).union(set(string.punctuation))
otros_ruidos = set(["''", "'d", "'ll", "'m", "'re", "'s", "'t", "'ve", '--', '000', '1', '2', '3', '4', '5', '6', '8', '10', '15', '30', 'I', 'F', '``', 'also', "don'", 'n', 'one', 'said', 'say', 'says', 'u', 'us'])

# Formas gramaticales de "hard" que queremos eliminar
formas_hard_palabras = set([w for (w, _) in formas_hard.keys()])

# Unir todos los filtros
palabras_omitidas = stop_words.union(otros_ruidos).union(formas_hard_palabras)

# Recoger palabras válidas del contexto
tokens_hard = []
for inst in hard_instances:
    for elemento in inst.context:
        if isinstance(elemento, tuple) and len(elemento) == 2:
            word, tag = elemento
            word_lower = word.lower()
            if word_lower not in palabras_omitidas:
                tokens_hard.append(word_lower)

# Calcular las 250 palabras más frecuentes
fdist_hard = FreqDist(tokens_hard)
vocabulario_hard = list(fdist_hard)[:250]

print("Primeras 10 palabras del vocabulario para 'hard':")
print(vocabulario_hard[:10])


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gran_\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Primeras 10 palabras del vocabulario para 'hard':
['may', 'lose', 'popular', 'support', 'someone', 'kill', 'defeat', 'clever', 'white', 'house']


Parte 6: Función para extraer el vector de características

El código define una función que genera un vector de características de bolsa de palabras para una instancia de "hard", indicando la presencia/ausencia de las palabras del vocabulario vocabulario_hard en su contexto. La salida muestra que las palabras "may", "lose", "popular", "support", "someone", "kill" y "defeat" están presentes en el contexto de la primera instancia, lo que sugiere un contexto semánticamente relevante para la WSD. Este vector es clave para el clasificador Naive Bayes basado en palabras vecinas.

In [17]:
def vector_caracteristicas_vecinas(instancia, vocabulario):
    contexto = set()
    for elemento in instancia.context:
        if isinstance(elemento, tuple) and len(elemento) == 2:
            word, _ = elemento
            contexto.add(word.lower())
    
    caracteristicas = {}
    for palabra in vocabulario:
        caracteristicas[f"contains({palabra})"] = (palabra in contexto)
    return caracteristicas

# Ejemplo con la primera instancia de "hard"
ejemplo_instancia = hard_instances[0]
vector = vector_caracteristicas_vecinas(ejemplo_instancia, vocabulario_hard)

# Mostrar solo las primeras 10 características para verificar
print("Vector de características (parcial):")
for i, (k, v) in enumerate(vector.items()):
    if i >= 10:
        break
    print(f"{k}: {v}")


Vector de características (parcial):
contains(may): True
contains(lose): True
contains(popular): True
contains(support): True
contains(someone): True
contains(kill): True
contains(defeat): True
contains(clever): False
contains(white): False
contains(house): False


Paso 7: Extracción de características basada en colocaciones (n-gramas)

El código define una función que genera un vector de características basado en bigramas (colocaciones) para la primera instancia de "hard" en el corpus Senseval 2. La salida muestra los bigramas "that 's" (anterior) y "to do" (posterior), que son relevantes para el contexto de la instancia, aunque "that 's" es menos informativo. Este vector es clave para el clasificador Naive Bayes basado en colocaciones, que superó al modelo de bolsa de palabras en el laboratorio. Las mejoras sugeridas podrían aumentar la robustez y especificidad de las características.

In [18]:
def vector_colocaciones(instancia, n=2):
    contexto = list(instancia.context)
    pos_objetivo = instancia.position  # posición de la palabra ambigua
    palabras = [w.lower() for (w, t) in contexto if isinstance((w, t), tuple)]

    caracteristicas = {}

    # Bigramas anteriores
    if pos_objetivo >= n:
        anterior = " ".join(palabras[pos_objetivo - n:pos_objetivo])
        caracteristicas[f"previous({anterior})"] = True
    else:
        anterior = " ".join(palabras[:pos_objetivo])
        caracteristicas[f"previous({anterior})"] = True

    # Bigramas posteriores
    if pos_objetivo + n < len(palabras):
        posterior = " ".join(palabras[pos_objetivo + 1:pos_objetivo + 1 + n])
        caracteristicas[f"next({posterior})"] = True
    else:
        posterior = " ".join(palabras[pos_objetivo + 1:])
        caracteristicas[f"next({posterior})"] = True

    return caracteristicas

# Prueba con la primera instancia de "hard"
vector_col = vector_colocaciones(hard_instances[0])

print("Vector de colocaciones:")
for k, v in vector_col.items():
    print(f"{k}: {v}")


Vector de colocaciones:
previous(that 's): True
next(to do): True


Paso 8: Preparar funciones de extracción de características

Este código define dos funciones para generar vectores de características para el laboratorio de WSD:

•	extraer_vecinos: Crea un vector de bolsa de palabras con 250 características (presencia/ausencia de palabras del vocabulario), similar a la función vector_caracteristicas_vecinas pero más concisa.
•	extraer_colocacion: Crea un vector con hasta dos características (bigramas anterior y posterior), similar a vector_colocaciones pero sin normalización a minúsculas.

Ambas funciones son esenciales para los clasificadores Naive Bayes, con las colocaciones ofreciendo mejor rendimiento debido a su especificidad. Las mejoras sugeridas (normalización, POS tags, filtrado de stopwords) podrían aumentar la robustez y efectividad de los vectores, alineándose con las propuestas del laboratorio.

In [19]:
def extraer_vecinos(instancia, vocabulario):
    contexto = [w.lower() for (w, t) in instancia.context if isinstance((w, t), tuple)]
    return {f"contains({pal})": (pal in contexto) for pal in vocabulario}

def extraer_colocacion(instancia, n=2):
    contexto = list(instancia.context)
    pos = instancia.position
    contexto_palabras = [w for (w, t) in contexto if isinstance((w, t), tuple)]

    prev = " ".join(contexto_palabras[max(0, pos - n):pos])
    next_ = " ".join(contexto_palabras[pos + 1:pos + 1 + n])

    features = {}
    if prev:
        features[f"previous({prev})"] = True
    if next_:
        features[f"next({next_})"] = True
    return features


Paso 9: Entrenamiento y evaluación de clasificadores para "hard"

Este código filtra instancias mal formateadas de "hard", divide los datos en entrenamiento (80%) y prueba (20%), entrena dos clasificadores Naive Bayes (uno con palabras vecinas, otro con colocaciones), y evalúa su rendimiento. La salida confirma las exactitudes del laboratorio (0.7891 para palabras vecinas, 0.8818 para colocaciones) y muestra una matriz de confusión que destaca los errores en sentidos menos frecuentes (HARD2, HARD3) debido al desbalance. Este código es la culminación de las etapas de limpieza, extracción de características y clasificación del laboratorio, validando la superioridad de las colocaciones.

In [20]:
from nltk.classify import NaiveBayesClassifier
from nltk.classify.util import accuracy
from nltk import ConfusionMatrix
import random

# Filtramos instancias mal formateadas
hard_filtrado = [i for i in hard_instances if isinstance(i.context, list) and all(isinstance(x, tuple) and len(x) == 2 for x in i.context)]

# Barajamos aleatoriamente y separamos 80/20
random.seed(42)
random.shuffle(hard_filtrado)
cutoff = int(0.8 * len(hard_filtrado))
train_hard = hard_filtrado[:cutoff]
test_hard = hard_filtrado[cutoff:]

# Entrenamiento con palabras vecinas
train_vecinos = [(extraer_vecinos(i, vocabulario_hard), i.senses[0]) for i in train_hard]
test_vecinos = [(extraer_vecinos(i, vocabulario_hard), i.senses[0]) for i in test_hard]

clf_vecinos = NaiveBayesClassifier.train(train_vecinos)
print("Exactitud (vecinos) - 'hard':", accuracy(clf_vecinos, test_vecinos))

# Entrenamiento con colocaciones
train_coloc = [(extraer_colocacion(i), i.senses[0]) for i in train_hard]
test_coloc = [(extraer_colocacion(i), i.senses[0]) for i in test_hard]

clf_coloc = NaiveBayesClassifier.train(train_coloc)
print("Exactitud (colocación) - 'hard':", accuracy(clf_coloc, test_coloc))

# Matriz de confusión para colocación
truth = [label for (feats, label) in test_coloc]
preds = [clf_coloc.classify(feats) for (feats, label) in test_coloc]
print("\nMatriz de confusión (colocación) - 'hard':")
print(ConfusionMatrix(truth, preds))


Exactitud (vecinos) - 'hard': 0.7891077636152954
Exactitud (colocación) - 'hard': 0.8818076477404403

Matriz de confusión (colocación) - 'hard':
      |   H   H   H |
      |   A   A   A |
      |   R   R   R |
      |   D   D   D |
      |   1   2   3 |
------+-------------+
HARD1 |<673>  4   9 |
HARD2 |  43 <58>  3 |
HARD3 |  39   4 <30>|
------+-------------+
(row = reference; col = test)



Paso 10: Entrenamiento y evaluación de clasificadores para "serve"

Este código filtra instancias de "serve", construye un vocabulario, genera vectores de características (palabras vecinas y colocaciones), entrena clasificadores Naive Bayes, y evalúa su rendimiento. La salida confirma las exactitudes del laboratorio (0.6713 para palabras vecinas, 0.7282 para colocaciones) y muestra una matriz de confusión que destaca los desafíos del desbalance y la clasificación de SERVE6. Las mejoras sugeridas podrían optimizar el vocabulario y la robustez del modelo.

In [21]:
from nltk.classify import NaiveBayesClassifier
from nltk.classify.util import accuracy
from nltk.metrics import ConfusionMatrix
import random

# Filtramos las instancias bien formateadas para 'serve'
serve_clean = [i for i in serve_instances if all(isinstance(t, tuple) and len(t) == 2 for t in i.context)]

# Construcción del vocabulario para "serve"
tokens_serve = []
for inst in serve_clean:
    for word, tag in inst.context:
        word_lower = word.lower()
        if word_lower not in palabras_omitidas:
            tokens_serve.append(word_lower)

fdist_serve = FreqDist(tokens_serve)
vocabulario_serve = list(fdist_serve)[:250]

# Función: características vecinas para "serve"
def vecinas_features_serve(instance):
    contexto = [w.lower() for (w, _) in instance.context if isinstance(_, str)]
    features = {}
    for palabra in vocabulario_serve:
        features[f'contains({palabra})'] = palabra in contexto
    return features

# Función: colocaciones para "serve"
def colocacion_features_serve(instance):
    contexto = list(instance.context)
    posicion = instance.position
    prev_bigram = ' '.join(w.lower() for (w, _) in contexto[max(0, posicion-2):posicion])
    next_bigram = ' '.join(w.lower() for (w, _) in contexto[posicion+1:posicion+3])
    return {
        f'previous({prev_bigram})': True,
        f'next({next_bigram})': True
    }

# Dataset con características vecinas
features_vecinas_serve = [(vecinas_features_serve(i), i.senses[0]) for i in serve_clean]
random.shuffle(features_vecinas_serve)
cut = int(len(features_vecinas_serve) * 0.8)
train_set_v, test_set_v = features_vecinas_serve[:cut], features_vecinas_serve[cut:]

# Dataset con colocaciones
features_colocacion_serve = [(colocacion_features_serve(i), i.senses[0]) for i in serve_clean]
random.shuffle(features_colocacion_serve)
cut = int(len(features_colocacion_serve) * 0.8)
train_set_c, test_set_c = features_colocacion_serve[:cut], features_colocacion_serve[cut:]

# Entrenamiento
clf_vecinas_serve = NaiveBayesClassifier.train(train_set_v)
clf_colocacion_serve = NaiveBayesClassifier.train(train_set_c)

# Evaluación
acc_vecinas_serve = accuracy(clf_vecinas_serve, test_set_v)
acc_colocacion_serve = accuracy(clf_colocacion_serve, test_set_c)

print(f"Exactitud (vecinos) - 'serve': {acc_vecinas_serve}")
print(f"Exactitud (colocación) - 'serve': {acc_colocacion_serve}")

# Matriz de confusión para colocación
gold = [label for (_, label) in test_set_c]
pred = [clf_colocacion_serve.classify(features) for (features, _) in test_set_c]
print("\nMatriz de confusión (colocación) - 'serve':")
print(ConfusionMatrix(gold, pred))


Exactitud (vecinos) - 'serve': 0.6713124274099884
Exactitud (colocación) - 'serve': 0.7282229965156795

Matriz de confusión (colocación) - 'serve':
        |   S   S         |
        |   E   E   S   S |
        |   R   R   E   E |
        |   V   V   R   R |
        |   E   E   V   V |
        |   1   1   E   E |
        |   0   2   2   6 |
--------+-----------------+
SERVE10 |<307> 17   7   4 |
SERVE12 |  57<173> 16   2 |
 SERVE2 |  40  21<121>  1 |
 SERVE6 |  60   3   6 <26>|
--------+-----------------+
(row = reference; col = test)



Paso 11: Instancias mal clasificadas para hard con colocaciones

Este código define una función es_valida que verifica si los contextos de las instancias de "hard" están bien formateados (tuplas de dos elementos) y filtra las instancias válidas en hard_clean, eliminando las 19 instancias con errores como "FRASL". Este paso es crucial para la limpieza de datos en el laboratorio de WSD, asegurando que las etapas de extracción de características y entrenamiento usen datos válidos. La implementación es modular y reutilizable, pero podría beneficiarse de validaciones adicionales y reportes de filtrado.

In [22]:
# Filtramos las instancias bien formateadas (sin 'FRASL' en el contexto)
def es_valida(inst):
    return all(isinstance(elem, tuple) and len(elem) == 2 for elem in inst.context)

hard_clean = [inst for inst in hard_instances if es_valida(inst)]

Paso 12: Instancias de 'hard' con sentido 'HARD1' mal clasificadas

Este código analiza instancias de "hard" con sentido HARD1 mal clasificadas por el clasificador Naive Bayes basado en colocaciones, pero la salida indica que no hay errores, lo cual contradice la matriz de confusión previa (13 errores). Esto sugiere una posible desalineación de datos o una división diferente. El código es útil para inspeccionar errores, pero requiere verificaciones de consistencia. La alta precisión de HARD1 es consistente con su frecuencia, pero el análisis de errores es clave para mejorar el modelo, como se propuso en el laboratorio.

In [23]:
print("Instancias de 'hard' con sentido 'HARD1' mal clasificadas:\n")

encontradas = 0
for inst, (features, label_gold) in zip(hard_clean[cut:], test_set_c):
    if label_gold == 'HARD1':
        pred = clf_colocacion.classify(features)
        if pred != 'HARD1':
            encontradas += 1
            print("Contexto:", " ".join(word for word, tag in inst.context))
            print(f"Sentido real: {label_gold}, Sentido predicho: {pred}")
            print("-" * 80)

if encontradas == 0:
    print("Ninguna instancia 'HARD1' fue mal clasificada en este conjunto de prueba.")



Instancias de 'hard' con sentido 'HARD1' mal clasificadas:

Ninguna instancia 'HARD1' fue mal clasificada en este conjunto de prueba.


CONCLUSIONES SOBRE EL USO DE APRENDIZAJE AUTOMÁTICO SUPERVISADO PARA DESAMBIGUAR EL SENTIDO DE LAS PALABRAS

A continuación las conclusiones obtenidas con base en el desarrollo de este lboratorio:

Los clasificadores dependen en gran medida de la calidad y el balance del corpus. Por ejemplo, si ciertos sentidos están sobrerrepresentados (como HARD1 o SERVE10), el clasificador tenderá a sesgarse hacia ellos.

Las características utilizadas (palabras vecinas y colocación) no capturan completamente la semántica profunda del lenguaje, ya que no tienen en cuenta relaciones semánticas o estructuras sintácticas más complejas.

No se manejan adecuadamente los contextos ambiguos o las frases con poca información contextual útil.

La desambiguación está limitada a los sentidos definidos en el corpus y no puede generalizar a nuevos sentidos no vistos.

Como alternativas para superar esas limitaciones y obtener un algoritmo que resuelva mejor el problema de la desambiguación del sentido de las palabras proponemos usar modelos de lenguaje modernos basados en redes neuronales como BERT o RoBERTa que pueden capturar relaciones semánticas complejas dentro del contexto. Además incorporar información sintáctica adicional (dependencias gramaticales) para enriquecer el vector de características junto con la aplicación de técnicas de aprendizaje no supervisado para detectar sentidos no etiquetados previamente.

Algo que también genera valor es expandir el corpus con más ejemplos para los sentidos menos representados y aplicar técnicas de balanceo de datos y finalmente usar enfoques híbridos que combinen características lingüísticas con embeddings semánticos o contexto dinámico.

A continuación la comparación de los Clasificadores:

Palabra Ambigua	Tipo de Clasificador	Exactitud
hard	Basado en palabras vecinas	0.7891
hard	Basado en colocación	0.8818
serve	Basado en palabras vecinas	0.6713
serve	Basado en colocación	0.7282

Reflexión adicional
¿Por qué no es justo comparar directamente la exactitud aportada por los clasificadores que han aprendido diferentes palabras ambiguas?

Porque la dificultad de desambiguar una palabra depende del número de sentidos que tenga, la similitud entre ellos y la cantidad de ejemplos disponibles para cada sentido. Por ejemplo, serve tiene más sentidos y puede tener contextos más ambiguos que hard.